In [1]:
import pyexasol
import configparser

config = configparser.ConfigParser()
config.read('C:\\Users\\svi02\\.spyder-py3\\ExasolConnect.ini')
dsn=config['exasolIUT']['dsn']
user=config['exasolIUT']['user']
pwd=config['exasolIUT']['pwd']
schema=config['exasolIUT']['schema']

connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
print("Connection Success")

Connection Success


In [2]:
#conda install -c conda-forge python-levenshtein
#pip install fuzzywuzzy
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import pandas as pd
# matching_list = pd.read_excel('C:\\Users\\svi02\\Downloads\\Siemens_Benchmark needed (1).xlsx'
#                          , sheet_name = 'City-benchmark needed'
#                          #, skiprows = [0]
#                          #, usecols = raw_data_col
#                         )
matching_list = pd.read_csv("C:\\Users\\svi02\\Documents\\misc\\dat.csv" , 
                            sep=';', 
                            encoding='latin1')
matching_list.head()
#fuzz.ratio("Catherine M Gitau","Catherine Gitau")

,CITY
0,Ácija
1,Águeda
2,AlÁ¨s
3,AlcÁºdia
4,AlcobaÁ§a


In [3]:
fuzz.partial_ratio("Catherine M. Gitau","Catherine Gitau")

80

In [4]:
list_string = matching_list['CITY'].values.tolist()
len(list_string)

491

In [5]:
QUERY = connect.execute("select * from DWHBIL.REL_CITY_TO_SOURCING_DESTINATION")

# Importing data into a DataFrame
query_df = pd.DataFrame()
a=[]
for row in QUERY:
    a.append(row)
#print(len(a))
query_df = pd.DataFrame(a)
df_col_names = QUERY.col_names
query_df.columns = df_col_names
print(query_df.shape)

(14440, 2)


In [6]:
import time
start_time = time.time()
choices = query_df['SOURCING_DESTINATION'].unique()
possibilities = []

for string in list_string:
    #print(string)
    possibility = process.extract(string, choices, limit=5, scorer=fuzz.token_sort_ratio)
    possibilities.append(possibility)
end_time = time.time()
print(end_time - start_time)

56.899619340896606


In [7]:
len(possibilities)

491

In [8]:
temp_df = pd.DataFrame(possibilities)
temp_df['string'] = list_string
temp_df.columns = ['Option-1', 'Option-2', 'Option-3', 'Option-4', 'Option-5','matching_string']
temp_df.head()

,Option-1,Option-2,Option-3,Option-4,Option-5,matching_string
0,"(Écija, 100)","(Jaú, 67)","(IKEJA, 67)","(Ischia, 60)","(Porcia, 60)",Ácija
1,"(Águeda, 100)","(Gouda, 80)","(Umea, 67)","(Guam, 67)","(Edam, 67)",Águeda
2,"(Alès, 86)","(Al Marsá, 73)","(Vaals, 67)","(Aalst, 67)","(?d?s 'Alem, 67)",AlÁ¨s
3,"(Alcúdia, 100)","(Talca, 73)","(Valencia, 71)","(Valdivia, 71)","(Palencia, 71)",AlcÁºdia
4,"(Alcobaça, 80)","(Alia?a, 71)","(Cordoba, AR, 67)","(Alba, 67)","(Valdosta, GA, 63)",AlcobaÁ§a


In [ ]:
temp_df.to_excel('C:\\Users\\svi02\\Downloads\\Matching_list.xlsx', 
                  sheet_name = 'Match',
                  header = True,
                  encoding='utf-8',
                  index=False)

In [ ]:
match_df = pd.DataFrame()
for possible in possibilities:
    match_df = match_df.append(possible, ignore_index=True)
match_df.columns = ['matching_city', 'fuzzy_score']
match_df = match_df[match_df['fuzzy_score'] > 80]
result = pd.merge(left = query_df, right = match_df, left_on = 'HOTEL_CITY_NAME', right_on = 'matching_city', how = 'right')

print(result)

In [ ]:
def match_name(name, list_names, min_score=0):
    # -1 score incase we don't get any matches
    max_score = -1
    # Returning empty name for no match as well
    max_name = ""
    # Iternating over all names in the other
    for name2 in list_names:
        #Finding fuzzy match score
        score = fuzz.ratio(name, name2)
        # Checking if we are above our threshold and have a better score
        if (score > min_score) & (score > max_score):
            max_name = name2
            max_score = score
    return (max_name, max_score)

In [ ]:
for name in list_string:
    print(match_name(name, df.HOTEL_CITY_NAME, min_score=90))

In [ ]:
def get_ratio(dataframe):
    name = dataframe['HOTEL_CITY_NAME']
    return fuzz.token_sort_ratio(name, "Indore")
print(get_ratio(df))
df[df.apply(get_ratio, axis=1) > 90]